In [ ]:
import polars as pl
from os import listdir
from datetime import date

In [ ]:
# Colab Drive desativado — dados locais em data/SoR · SoT · Spec
# from google.colab import drive
# drive.mount("/content/drive")


In [ ]:
# ============================================================
# CONFIGURAÇÕES DO PROJETO (monorepo local)
# SoR ≈ Bronze · SoT ≈ Silver · Spec ≈ Gold
# ============================================================
from pathlib import Path

_here = Path.cwd().resolve()
if (_here / "data" / "SoR").exists():
    _data = _here / "data"
elif (_here.parent / "data" / "SoR").exists():
    _data = _here.parent / "data"
else:
    _data = _here / "data"

pasta_projeto = str(_data)
pasta_sor = str(_data / "SoR")
pasta_sot = str(_data / "SoT")
pasta_spec = str(_data / "Spec")

pasta_voos_operacores = fr'{pasta_sor}/SoR_voos_operacoes'
pasta_tarifas = fr'{pasta_sor}/SoR_tarifas_aereas'


In [ ]:
hoje = date.today()
hoje_str = hoje.strftime('%Y%m%d')

#**1. PROCESSO DE ETL E MATERIALIZAÇÃO DA TABELA SILVER DE HISTÓRICO DE VÔOS**

In [ ]:
df_sor_voos = pl.read_csv(fr'{pasta_voos_operacores}/*.csv', separator=';', ignore_errors=True,
truncate_ragged_lines=True, skip_rows=1, has_header=True)

print(df_sor_voos.shape)
df_sor_voos.head()

In [ ]:
colunas_voos = {
 'ICAO Empresa Aérea':'icao_empresa',
 'Número Voo':'numero_voo',
 'Código Autorização (DI)':'codigo_autorizacao',
 'Código Tipo Linha':'codigo_tipo_linha',
 'ICAO Aeródromo Origem':'icao_aerodromo_origem',
 'ICAO Aeródromo Destino':'icao_aerodromo_destino',
 'Partida Prevista':'partida_prevista',
 'Partida Real':'partida_real',
 'Chegada Prevista':'chegada_prevista',
 'Chegada Real':'chegada_real',
 'Situação Voo':'situacao_voo',
 'Código Justificativa':'codigo_justificativa'
}

In [ ]:
df_icao = pl.read_csv('https://davidmegginson.github.io/ourairports-data/airports.csv',
                      columns=['icao_code','iso_country','municipality','name']).rename(
                          {'iso_country':'pais', 'municipality':'municipio','name':'aeroporto'}).unique()
print(df_icao.shape)
df_icao.head()

In [ ]:
df_empresas = pl.read_csv(
    'https://raw.githubusercontent.com/jpatokal/openflights/master/data/airlines.dat',
    has_header=False,
    new_columns=["id", "nome_empresa", "alias", "iata", "icao", "callsign", "pais", "ativa"]).select([
    pl.col("icao").alias("icao_empresa"),
    pl.col("nome_empresa"),
])
df_empresas.head()


##**1.1 Agregação e Tratamento de Dados**

In [ ]:
df_sot_voos = df_sor_voos.with_columns(
    pl.col(['Partida Prevista', 'Partida Real', 'Chegada Prevista', 'Chegada Real'])
    .str.to_datetime(format='%Y-%m-%d %H:%M:%S', strict=False)
    ).join(df_icao, left_on='ICAO Aeródromo Origem', right_on='icao_code', how='left').join(
        df_icao, left_on='ICAO Aeródromo Destino', right_on='icao_code', how='left').rename(
        {'pais':'pais_origem', 'municipio':'municipio_origem', 'pais_right':'pais_destino', 'municipio_right':'municipio_destino'
        ,'aeroporto':'aeroporto_origem', 'aeroporto_right':'aeroporto_destino'}).join(
            df_empresas, how='left', left_on='ICAO Empresa Aérea', right_on='icao_empresa'
          ).filter((pl.col('pais_origem')=='BR') & (pl.col('pais_destino')=='BR')).rename(colunas_voos)

print(df_sot_voos.shape)
df_sot_voos.head()

In [ ]:
df_sot_voos.select('situacao_voo').unique()

##**1.2 Cálculos de Datas, Atrasos e Status**

In [ ]:
df_sot_voos = df_sot_voos.with_columns([
    ((pl.col('chegada_prevista') - pl.col('partida_prevista')).dt.total_minutes().cast(pl.Float64) / 60).alias('duracao_prevista'),
    ((pl.col('chegada_real') - pl.col('partida_real')).dt.total_minutes().cast(pl.Float64) / 60).alias('duracao_real'),

    pl.when(pl.col('situacao_voo') == 'REALIZADO').then(pl.lit(1))
    .when(pl.col('situacao_voo') == 'CANCELADO').then(pl.lit(-1))
    .otherwise(pl.lit(0))
    .alias('status_situacao_voo'),

    ((pl.col('partida_prevista') - pl.col('partida_real')).dt.total_minutes().cast(pl.Float64) / 60).alias('dif_partida'),
    ((pl.col('chegada_prevista') - pl.col('chegada_real')).dt.total_minutes().cast(pl.Float64) / 60).alias('dif_chegada')
]).with_columns(
    pl.when(pl.col('dif_partida')<0).then(pl.lit(-1))
    .when(pl.col('dif_partida')>0).then(pl.lit(1))
    .otherwise(pl.lit(0)).alias('status_partida'),

    pl.when(pl.col('dif_partida')<0).then(pl.lit('atrasado'))
    .when(pl.col('dif_partida')>0).then(pl.lit('adiantado'))
    .otherwise(pl.lit('em_tempo')).alias('desc_partida'),

    pl.when(pl.col('dif_chegada')<0).then(pl.lit(-1))
    .when(pl.col('dif_chegada')>0).then(pl.lit(1))
    .otherwise(pl.lit(0)).alias('status_chegada'),

    pl.when(pl.col('dif_chegada')<0).then(pl.lit('atrasado'))
    .when(pl.col('dif_chegada')>0).then(pl.lit('adiantado'))
    .otherwise(pl.lit('em_tempo')).alias('desc_chegada'),

    pl.lit(hoje).alias('data_referencia')
    )

print(df_sot_voos.shape)
df_sot_voos.head()

In [ ]:
df_sot_voos.schema

##**1.3 Materialização da tabela de histórico de vôos nacionais**

In [ ]:
df_sot_voos.write_parquet(fr'{pasta_sot}/SoT_historico_voos/historico_voos_full_{hoje_str}.parquet')

##**1.4 Histórico de origens e destinos & percentual de vôos realizados e cancelados.**

In [ ]:
df_a = df_sot_voos.filter(pl.col('status_situacao_voo')==1).group_by(['municipio_origem','municipio_destino']).agg(
    pl.col('numero_voo').len().cast(pl.Int64).alias('voos_realizados'))

df_b = df_sot_voos.filter(pl.col('status_situacao_voo')==-1).group_by(['municipio_origem','municipio_destino']).agg(
    pl.col('numero_voo').len().cast(pl.Int64).alias('voos_cancelados'))

df_spec_orig_destino = df_a.join(df_b, on=['municipio_origem','municipio_destino'], how='left').fill_null(0).with_columns(
    (pl.col('voos_cancelados')/pl.sum_horizontal(['voos_realizados','voos_cancelados'])).round(4).alias('pct_no_show'),
    pl.lit(hoje).alias('data_referencia')
).sort('voos_realizados', descending=True)

df_spec_orig_destino.write_parquet(fr'{pasta_spec}/spec_origens_destinos/historico_origens_destinos_{hoje_str}.parquet')

print(df_spec_orig_destino.shape)
df_spec_orig_destino.head()

In [ ]:
df_spec_orig_destino.schema

##**1.5 Histórico de datas (mês e dia) & percentual de vôos realizados e cancelados.**

In [ ]:
df_a = df_sot_voos.filter(pl.col('status_situacao_voo')==1).with_columns(
    pl.col('partida_real').dt.month().alias('n_mes'),
    pl.col('partida_real').dt.day().alias('n_dia')).group_by(['n_mes','n_dia']).agg(
    pl.col('numero_voo').len().cast(pl.Int64).alias('voos_realizados'))

df_b = df_sot_voos.filter(pl.col('status_situacao_voo')==-1).with_columns(
    pl.col('partida_prevista').dt.month().alias('n_mes'),
    pl.col('partida_prevista').dt.day().alias('n_dia')).group_by(['n_mes','n_dia']).agg(
    pl.col('numero_voo').len().cast(pl.Int64).alias('voos_cancelados'))

df_spec_datas = df_a.join(df_b, on=['n_mes','n_dia'], how='left').fill_null(0).with_columns(
    (pl.col('voos_cancelados')/pl.sum_horizontal(['voos_realizados','voos_cancelados'])).round(4).alias('pct_no_show'),
    pl.lit(hoje).alias('data_referencia')
).sort('pct_no_show', descending=True)

df_spec_datas.write_parquet(fr'{pasta_spec}/spec_datas/historico_datas_{hoje_str}.parquet')

print(df_spec_datas.shape)
df_spec_datas.head()

In [ ]:
df_spec_datas.schema

##**1.6 Histórico de destinos & percentual de vôos com atraso.**

In [ ]:
#df_sot_datas_gb = df_sot_voos.filter(pl.col('desc_partida')=='atrasado').with_columns(
df_a = df_sot_voos.filter((pl.col('status_situacao_voo')==1)).group_by(
    ['municipio_origem','municipio_destino']).agg(pl.col('numero_voo').len().cast(pl.Int64).alias('voos_totais'))

df_b = df_sot_voos.filter((pl.col('status_situacao_voo')==1) & (pl.col('desc_partida')=='atrasado')).group_by(
    ['municipio_origem','municipio_destino']).agg(pl.col('numero_voo').len().cast(pl.Int64).alias('voos_atrasados'))

df_spec_atraso = df_a.join(df_b, on=['municipio_origem','municipio_destino'], how='left').fill_null(0).with_columns(
    (pl.col('voos_atrasados')/pl.col('voos_totais')).round(4).alias('pct_atraso'),
    pl.lit(hoje).alias('data_referencia')
).sort('pct_atraso', descending=True)

df_spec_atraso.write_parquet(fr'{pasta_spec}/spec_atrasos/historico_atrasos_{hoje_str}.parquet')

print(df_spec_atraso.shape)
df_spec_atraso.head()

In [ ]:
df_spec_atraso.schema

##**1.7 Histórico de cancelamentos e atrasos por companhia aérea.**

In [ ]:
df_a = df_sot_voos.filter(pl.col('nome_empresa').is_not_null()
).group_by('nome_empresa').agg(pl.col('numero_voo').len().cast(pl.Int64).alias('voos_totais'))

df_b = df_sot_voos.filter(pl.col('status_situacao_voo')==-1).group_by('nome_empresa').agg(
    pl.col('numero_voo').len().cast(pl.Int64).alias('voos_cancelados'))

df_c = df_sot_voos.filter((pl.col('status_situacao_voo')==1) & (pl.col('desc_partida')=='atrasado')).group_by('nome_empresa').agg(
    pl.col('numero_voo').len().cast(pl.Int64).alias('voos_atrasados'))

df_spec_companhias = df_a.join(df_b, on='nome_empresa', how='left').join(df_c, on='nome_empresa', how='left').fill_null(0).with_columns(
    (pl.col('voos_cancelados')/pl.col('voos_totais')).round(4).alias('pct_cancelamento'),
    (pl.col('voos_atrasados')/pl.col('voos_totais')).round(4).alias('pct_atraso'),
    pl.lit(hoje).alias('data_referencia')).sort('voos_totais', descending=True)

df_spec_companhias.write_parquet(fr'{pasta_spec}/spec_companhias/historico_companhias_{hoje_str}.parquet')

print(df_spec_companhias.shape)
df_spec_companhias.head(10)

In [ ]:
df_spec_companhias.schema

#**2. PROCESSO DE ETL E MATERIALIZAÇÃO DA TABELA SILVER DE HISTÓRICO DE TARIFAS**

In [ ]:
ls_arquivos = []
for arq in listdir(pasta_tarifas):
  df = pl.read_csv(fr'{pasta_tarifas}/{arq}', separator=';', ignore_errors=True, encoding='iso-8859-1')
  df = df.rename({
  'Ano de Referência':'ano','Mês de Referência':'n_mes','ICAO Empresa Aérea':'empresa','ICAO Aeródromo Origem':'origem','ICAO Aeródromo Destino':'destino','Tarifa-N':'tarifa','Assentos Comercializados':'assentos',
  'nr_ano_referencia':'ano','nr_mes_referencia':'n_mes','sg_empresa_icao':'empresa','sg_icao_origem':'origem','sg_icao_destino':'destino','nr_tarifa':'tarifa','nr_assentos':'assentos',
  'ANO':'ano','MES':'n_mes','EMPRESA':'empresa','ORIGEM':'origem','DESTINO':'destino','TARIFA':'tarifa','ASSENTOS':'assentos'}
  , strict=False)

  df= df.with_columns(
      pl.col('tarifa').cast(pl.String, strict=False).str.replace(',', '.')).with_columns(
    pl.col('ano').cast(pl.Int16),
    pl.col('n_mes').cast(pl.Int8),
    pl.col('tarifa').cast(pl.Float64),
    pl.col('assentos').cast(pl.Int64),
    pl.lit(hoje).alias('data_referencia')).select(['ano','n_mes','empresa','origem','destino','tarifa','assentos'])

  df = df.join(df_empresas, how='left', left_on='empresa', right_on='icao_empresa').join(
    df_icao, how='left', left_on='origem', right_on='icao_code').join(
    df_icao, how='left', left_on='destino', right_on='icao_code').rename({
        'aeroporto':'aeroporto_origen','pais':'pais_origem','municipio':'municipio_origem',
        'aeroporto_right':'aeroporto_destino','pais_right':'pais_destino','municipio_right':'municipio_destino'
    }).filter((pl.col('pais_origem')=='BR') & (pl.col('pais_destino')=='BR'))

  df.write_parquet(fr'{pasta_sot}/SoT_tarifas/tarifas_{arq}_{hoje_str}.parquet')

  ls_arquivos.append(df)

In [ ]:
df_tarifas_sot = pl.concat(ls_arquivos)

print(df_tarifas_sot.shape)
df_tarifas_sot.head()

In [ ]:
df_tarifas_sot.schema

#**3. MATERIALIZAÇÃO DE TABELA SILVER DE AEROPORTOS**

In [ ]:
df_aeroportos = (pl.read_csv("https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv")
    .filter(pl.col("iso_country") == "BR")
    .filter(pl.col("iata_code").is_not_null())
    .select([
        pl.col("icao_code").alias("codigo_icao"),
        pl.col("iata_code").alias("codigo_iata"),
        pl.col("municipality").alias("municipio"),
        pl.col("name").alias("nome_aeroporto"),
        pl.col("iso_country").alias("pais"),
        pl.col("scheduled_service").alias("servico_regular"),
    ]))

print(df_aeroportos.shape)
df_aeroportos.head()


In [ ]:
df_aeroportos.write_parquet(fr'{pasta_sot}/SoT_aeroportos/aeroportos_{hoje_str}.parquet')